# Aneurysm Volume Prediction v4 (Attention + Stack Calibration)

目标：在 v2_attention_plus(0.7976) 基础上继续提分。

核心改进：
1. AttentionUNet3D + SE + GroupNorm（小batch稳定）
2. 5折分层CV + 多seed模型集成
3. TTA推理
4. 指标定向优化：多阈值体积特征工程 + 线性堆叠校准（stacked calibration）
5. 平台路径强适配，确保提交文件写入评测器路径


In [ ]:
# 如缺包请取消注释
# %pip install nibabel scikit-learn tqdm pandas matplotlib
# %pip install torch torchvision torchaudio


In [ ]:
import os
import sys
import json
import random
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import nibabel as nib
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LinearRegression, Ridge

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
torch.backends.cudnn.benchmark = True

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
USE_AMP = (DEVICE.type == "cuda")

print("Python:", sys.executable)
print("Torch:", torch.__version__)
print("Device:", DEVICE, "AMP:", USE_AMP)


In [ ]:
# ===== 跨平台路径自动识别 =====
DATA_ROOT = os.environ.get("ANEURYSM_DATA_ROOT", "").strip()

def is_valid_dataset_dir(p: Path) -> bool:
    return (p / "train").exists() and (p / "test").exists() and (p / "train_labels").exists()

def find_base_dir():
    if DATA_ROOT:
        p = Path(DATA_ROOT)
        if is_valid_dataset_dir(p):
            return p
    fixed = [
        "/dataset/public",
        "/dataset",
        "/kaggle/input/aneurysm-volume-prediction",
        "/kaggle/input/aneurysm-volume",
        "/Users/songling/Desktop/Aneurysm Volume Prediction",
    ]
    for x in fixed:
        p = Path(x)
        if is_valid_dataset_dir(p):
            return p
    for root in [Path("/dataset"), Path("/kaggle/input"), Path.cwd()]:
        if not root.exists():
            continue
        for d in root.rglob("*"):
            if d.is_dir() and is_valid_dataset_dir(d):
                return d
    return None

BASE_DIR = find_base_dir()
if BASE_DIR is None:
    raise FileNotFoundError("未找到数据目录，请设置 ANEURYSM_DATA_ROOT")

TRAIN_IMG_DIR = BASE_DIR / "train"
TRAIN_MASK_DIR = BASE_DIR / "train_labels"
TEST_IMG_DIR = BASE_DIR / "test"
TRAIN_CSV = BASE_DIR / "train.csv"

if Path("/root/setup/solution/working").exists():
    OUTPUT_ROOT = Path("/root/setup/solution/working")
elif Path("/working").exists():
    OUTPUT_ROOT = Path("/working")
elif Path("/kaggle/working").exists():
    OUTPUT_ROOT = Path("/kaggle/working")
else:
    OUTPUT_ROOT = BASE_DIR / "working"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

RUN_NAME = os.environ.get("RUN_NAME", "").strip() or datetime.now().strftime("v4_stack_%Y%m%d_%H%M%S")
EXP_DIR = OUTPUT_ROOT / RUN_NAME
CKPT_DIR = EXP_DIR / "checkpoints"
PRED_DIR = EXP_DIR / "predictions"
LOG_DIR = EXP_DIR / "logs"
for d in [EXP_DIR, CKPT_DIR, PRED_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("EXP_DIR:", EXP_DIR)


In [ ]:
def load_nii(path):
    nii = nib.load(str(path))
    arr = nii.get_fdata(dtype=np.float32)
    spacing = nii.header.get_zooms()[:3]
    return arr, spacing

def robust_zscore(x, eps=1e-6):
    lo, hi = np.percentile(x, [0.5, 99.5])
    x = np.clip(x, lo, hi)
    m, s = x.mean(), x.std()
    return (x - m) / (s + eps)

def volume_from_binary(mask_zyx, spacing):
    return float(max(mask_zyx.sum() * float(spacing[0] * spacing[1] * spacing[2]), 0.0))

def volume_from_prob(prob_zyx, spacing):
    return float(max(prob_zyx.sum() * float(spacing[0] * spacing[1] * spacing[2]), 0.0))

def volumetric_similarity(y_true, y_pred, eps=1e-4):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.mean(1.0 - np.abs(y_true - y_pred) / (y_true + y_pred + eps)))

def parse_pid(path):
    return int(Path(path).name.split(".")[0])


In [ ]:
class AneurysmDataset(Dataset):
    def __init__(self, img_paths, mask_paths=None, augment=False):
        self.img_paths = img_paths
        self.mask_paths = mask_paths
        self.augment = augment

    def __len__(self):
        return len(self.img_paths)

    def _aug(self, img, m):
        if random.random() < 0.5:
            img = torch.flip(img, dims=[2]); m = torch.flip(m, dims=[2]) if m is not None else None
        if random.random() < 0.5:
            img = torch.flip(img, dims=[3]); m = torch.flip(m, dims=[3]) if m is not None else None
        if random.random() < 0.5:
            k = random.randint(0, 3)
            img = torch.rot90(img, k=k, dims=[2,3])
            if m is not None: m = torch.rot90(m, k=k, dims=[2,3])
        if random.random() < 0.7:
            img = img * (1.0 + random.uniform(-0.15,0.15)) + random.uniform(-0.12,0.12)
        if random.random() < 0.3:
            img = img + torch.randn_like(img) * 0.03
        return img, m

    def __getitem__(self, idx):
        p = self.img_paths[idx]
        x, sp = load_nii(p)
        x = robust_zscore(x)
        x = np.transpose(x, (2,0,1)).astype(np.float32)
        x = torch.from_numpy(x).unsqueeze(0)
        out = {"image": x, "spacing": torch.tensor(sp, dtype=torch.float32), "pid": parse_pid(p)}
        if self.mask_paths is not None:
            m, _ = load_nii(self.mask_paths[idx])
            m = (m > 0.5).astype(np.float32)
            m = np.transpose(m, (2,0,1)).astype(np.float32)
            m = torch.from_numpy(m).unsqueeze(0)
            if self.augment:
                x, m = self._aug(x, m)
                out["image"] = x
            out["mask"] = m
        return out


In [ ]:
def gn(ch):
    g = 8
    while ch % g != 0 and g > 1:
        g -= 1
    return nn.GroupNorm(g, ch)

class SE3D(nn.Module):
    def __init__(self, ch, r=8):
        super().__init__()
        hid = max(ch // r, 4)
        self.net = nn.Sequential(nn.AdaptiveAvgPool3d(1), nn.Conv3d(ch, hid, 1), nn.ReLU(inplace=True), nn.Conv3d(hid, ch, 1), nn.Sigmoid())
    def forward(self, x):
        return x * self.net(x)

class ResGNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, p_drop=0.1):
        super().__init__()
        self.c1 = nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False)
        self.n1 = gn(out_ch)
        self.c2 = nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False)
        self.n2 = gn(out_ch)
        self.se = SE3D(out_ch)
        self.drop = nn.Dropout3d(p_drop)
        self.act = nn.ReLU(inplace=True)
        self.proj = nn.Conv3d(in_ch, out_ch, 1, bias=False) if in_ch != out_ch else nn.Identity()
    def forward(self, x):
        i = self.proj(x)
        x = self.act(self.n1(self.c1(x)))
        x = self.n2(self.c2(x))
        x = self.se(x)
        x = self.drop(x)
        return self.act(x + i)

class AttnGate3D(nn.Module):
    def __init__(self, x_ch, g_ch, i_ch):
        super().__init__()
        self.wx = nn.Conv3d(x_ch, i_ch, 1, bias=False)
        self.wg = nn.Conv3d(g_ch, i_ch, 1, bias=False)
        self.psi = nn.Sequential(nn.ReLU(inplace=True), nn.Conv3d(i_ch, 1, 1), nn.Sigmoid())
    def forward(self, x, g):
        return x * self.psi(self.wx(x) + self.wg(g))

class AttentionUNet3D(nn.Module):
    def __init__(self, base=24):
        super().__init__()
        self.e1 = ResGNBlock(1, base)
        self.p1 = nn.MaxPool3d(2)
        self.e2 = ResGNBlock(base, base*2)
        self.p2 = nn.MaxPool3d(2)
        self.e3 = ResGNBlock(base*2, base*4)
        self.p3 = nn.MaxPool3d(2)
        self.b = ResGNBlock(base*4, base*8, p_drop=0.2)
        self.u3 = nn.ConvTranspose3d(base*8, base*4, 2, 2)
        self.a3 = AttnGate3D(base*4, base*4, base*2)
        self.d3 = ResGNBlock(base*8, base*4)
        self.u2 = nn.ConvTranspose3d(base*4, base*2, 2, 2)
        self.a2 = AttnGate3D(base*2, base*2, base)
        self.d2 = ResGNBlock(base*4, base*2)
        self.u1 = nn.ConvTranspose3d(base*2, base, 2, 2)
        self.a1 = AttnGate3D(base, base, max(base//2, 4))
        self.d1 = ResGNBlock(base*2, base)
        self.head = nn.Conv3d(base, 1, 1)
    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(self.p1(e1)); e3 = self.e3(self.p2(e2)); b = self.b(self.p3(e3))
        d3 = self.u3(b); d3 = self.d3(torch.cat([d3, self.a3(e3, d3)], dim=1))
        d2 = self.u2(d3); d2 = self.d2(torch.cat([d2, self.a2(e2, d2)], dim=1))
        d1 = self.u1(d2); d1 = self.d1(torch.cat([d1, self.a1(e1, d1)], dim=1))
        return self.head(d1)

class DiceBCELoss(nn.Module):
    def __init__(self, pos_weight=4.5, bce_weight=0.45, smooth=1e-5):
        super().__init__()
        self.pos_weight = pos_weight
        self.bce_weight = bce_weight
        self.smooth = smooth
    def forward(self, logit, target):
        pw = torch.tensor([self.pos_weight], device=logit.device, dtype=logit.dtype)
        bce = nn.functional.binary_cross_entropy_with_logits(logit, target, pos_weight=pw)
        p = torch.sigmoid(logit).reshape(logit.size(0), -1)
        t = target.reshape(target.size(0), -1)
        inter = (p*t).sum(dim=1)
        den = p.sum(dim=1) + t.sum(dim=1)
        dice = 1.0 - (2.0*inter + self.smooth) / (den + self.smooth)
        return self.bce_weight * bce + (1.0 - self.bce_weight) * dice.mean()


In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
train_df["patient_id"] = train_df["patient_id"].astype(int)
all_train_imgs = sorted(TRAIN_IMG_DIR.glob("*.nii.gz"))
all_train_msks = [TRAIN_MASK_DIR / p.name for p in all_train_imgs]
all_test_imgs = sorted(TEST_IMG_DIR.glob("*.nii.gz"))
print("train:", len(all_train_imgs), "test:", len(all_test_imgs))


In [ ]:
CFG = {
    "n_splits": 5,
    "epochs": 70,
    "batch_size": 2,
    "num_workers": 0,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "base_ch": 24,
    "grad_clip": 1.0,
    "tta": True,
    "early_stop_patience": 14,
    "seeds": [42, 2025, 3407],
}
print(CFG)


In [ ]:
def make_splits(train_df, n_splits=5, seed=42):
    vols = train_df.sort_values("patient_id")["volume"].values
    try:
        bins = pd.qcut(vols, q=5, labels=False, duplicates="drop")
    except Exception:
        bins = pd.cut(vols, bins=5, labels=False)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return list(skf.split(np.arange(len(all_train_imgs)), bins))

def infer_tta(model, x):
    model.eval(); outs=[]
    with torch.no_grad():
        p = torch.sigmoid(model(x)); outs.append(p)
        p = torch.sigmoid(model(torch.flip(x, dims=[3]))); outs.append(torch.flip(p, dims=[3]))
        p = torch.sigmoid(model(torch.flip(x, dims=[4]))); outs.append(torch.flip(p, dims=[4]))
        p = torch.sigmoid(model(torch.flip(x, dims=[3,4]))); outs.append(torch.flip(p, dims=[3,4]))
    return torch.mean(torch.stack(outs, 0), 0)

def train_fold(seed, fold, tr_idx, va_idx):
    seed_everything(seed + fold)
    tr_ds = AneurysmDataset([all_train_imgs[i] for i in tr_idx], [all_train_msks[i] for i in tr_idx], augment=True)
    va_ds = AneurysmDataset([all_train_imgs[i] for i in va_idx], [all_train_msks[i] for i in va_idx], augment=False)
    tr_ld = DataLoader(tr_ds, batch_size=CFG["batch_size"], shuffle=True, num_workers=CFG["num_workers"])
    va_ld = DataLoader(va_ds, batch_size=1, shuffle=False, num_workers=CFG["num_workers"])

    model = AttentionUNet3D(base=CFG["base_ch"]).to(DEVICE)
    crit = DiceBCELoss(pos_weight=4.5, bce_weight=0.45)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG["epochs"], eta_min=1e-5)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    sd = CKPT_DIR / f"seed_{seed}"
    sd.mkdir(parents=True, exist_ok=True)
    ck = sd / f"fold_{fold}.pt"
    best_vs, no_imp = -1.0, 0

    for ep in range(CFG["epochs"]):
        model.train(); losses=[]
        for b in tr_ld:
            x = b["image"].to(DEVICE); y = b["mask"].to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                logit = model(x); loss = crit(logit, y)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
            scaler.step(opt); scaler.update()
            losses.append(loss.item())
        sch.step()

        model.eval(); gts=[]; prs=[]
        with torch.no_grad():
            for b in va_ld:
                x = b["image"].to(DEVICE)
                m = b["mask"].cpu().numpy()[0,0]
                sp = b["spacing"].numpy()[0]
                p = torch.sigmoid(model(x)).cpu().numpy()[0,0]
                gts.append(volume_from_binary(m, sp))
                prs.append(volume_from_binary((p>0.5).astype(np.uint8), sp))
        vs = volumetric_similarity(gts, prs)
        print(f"seed {seed} fold {fold} ep {ep+1:02d}/{CFG["epochs"]} loss={np.mean(losses):.4f} valVS={vs:.4f}")
        if vs > best_vs:
            best_vs = vs; no_imp = 0; torch.save(model.state_dict(), ck)
        else:
            no_imp += 1
        if no_imp >= CFG["early_stop_patience"]:
            print(f"early stop seed {seed} fold {fold}, best={best_vs:.4f}")
            break
    return ck, best_vs

def collect_oof_probs(model_paths_by_seed, splits):
    items=[]
    for fold, (_, va_idx) in enumerate(splits):
        models=[]
        for seed, paths in model_paths_by_seed.items():
            m = AttentionUNet3D(base=CFG["base_ch"]).to(DEVICE)
            m.load_state_dict(torch.load(paths[fold], map_location=DEVICE))
            m.eval(); models.append(m)
        va_ds = AneurysmDataset([all_train_imgs[i] for i in va_idx], [all_train_msks[i] for i in va_idx], augment=False)
        va_ld = DataLoader(va_ds, batch_size=1, shuffle=False)
        with torch.no_grad():
            for b in va_ld:
                pid = int(b["pid"][0]) if isinstance(b["pid"], torch.Tensor) else int(b["pid"])
                x = b["image"].to(DEVICE)
                sp = b["spacing"].numpy()[0].astype(np.float32)
                probs=[]
                for m in models:
                    p = infer_tta(m, x).cpu().numpy()[0,0] if CFG["tta"] else torch.sigmoid(m(x)).cpu().numpy()[0,0]
                    probs.append(p)
                prob = np.mean(probs, axis=0).astype(np.float32)
                items.append({"pid": pid, "spacing": sp, "prob": prob})
    return items

def build_volume_features(prob, spacing, thr_list):
    feat = {}
    feat["soft"] = volume_from_prob(prob, spacing)
    feat["p_mean"] = float(prob.mean())
    feat["p_max"] = float(prob.max())
    feat["p_std"] = float(prob.std())
    for t in thr_list:
        feat[f"h_{t:.2f}"] = volume_from_binary((prob > t).astype(np.uint8), spacing)
    return feat


In [ ]:
start = time.time()
splits = make_splits(train_df, n_splits=CFG["n_splits"], seed=42)
model_paths_by_seed = {}
rows=[]
for seed in CFG["seeds"]:
    seed_paths=[]
    for fold, (tr_idx, va_idx) in enumerate(splits):
        p, s = train_fold(seed, fold, tr_idx, va_idx)
        seed_paths.append(p)
        rows.append({"seed":seed, "fold":fold, "best_vs":float(s), "ckpt":str(p)})
    model_paths_by_seed[seed]=seed_paths
pd.DataFrame(rows).to_csv(LOG_DIR / "fold_scores.csv", index=False)
print(f"training done in {(time.time()-start)/60:.1f} min")


In [ ]:
# ===== OOF 特征堆叠校准 =====
oof_items = collect_oof_probs(model_paths_by_seed, splits)
gt_map = {int(r.patient_id): float(r.volume) for _, r in train_df.iterrows()}
thr_list = [round(x,2) for x in np.arange(0.30, 0.71, 0.05)]

feat_rows=[]
for it in oof_items:
    pid = int(it["pid"])
    f = build_volume_features(it["prob"], it["spacing"], thr_list)
    f["patient_id"] = pid
    f["gt"] = gt_map[pid]
    feat_rows.append(f)

feat_df = pd.DataFrame(feat_rows).sort_values("patient_id").reset_index(drop=True)
feature_cols = [c for c in feat_df.columns if c not in ["patient_id", "gt"]]
X = feat_df[feature_cols].values
y = feat_df["gt"].values

# 基线：单阈值+软硬融合搜索
best = {"vs":-1, "thr":0.5, "alpha":1.0}
for thr in np.arange(0.30, 0.71, 0.01):
    hard = []
    soft = []
    for it in oof_items:
        sp = it["spacing"]; prob = it["prob"]
        hard.append(volume_from_binary((prob > thr).astype(np.uint8), sp))
        soft.append(volume_from_prob(prob, sp))
    hard = np.asarray(hard); soft = np.asarray(soft)
    for a in np.arange(0.0, 1.01, 0.1):
        pred = a * hard + (1-a) * soft
        vs = volumetric_similarity(y, pred)
        if vs > best["vs"]:
            best = {"vs":float(vs), "thr":float(thr), "alpha":float(a)}

# 元学习器1：positive linear regression
meta1 = LinearRegression(positive=True)
meta1.fit(X, y)
pred1 = np.clip(meta1.predict(X), 0, None)
vs1 = volumetric_similarity(y, pred1)

# 元学习器2：ridge（防止系数过拟合）
meta2 = Ridge(alpha=1.0)
meta2.fit(X, y)
pred2 = np.clip(meta2.predict(X), 0, None)
vs2 = volumetric_similarity(y, pred2)

# 线性校准基线
hard_best = np.array([volume_from_binary((it["prob"] > best["thr"]).astype(np.uint8), it["spacing"]) for it in oof_items])
soft_best = np.array([volume_from_prob(it["prob"], it["spacing"]) for it in oof_items])
base_pred = best["alpha"] * hard_best + (1-best["alpha"]) * soft_best
k, b = np.polyfit(base_pred, y, 1)
base_cal = np.clip(k * base_pred + b, 0, None)
vs_base = volumetric_similarity(y, base_pred)
vs_base_cal = volumetric_similarity(y, base_cal)

choices = [
    ("base", vs_base),
    ("base_cal", vs_base_cal),
    ("meta_linear_pos", vs1),
    ("meta_ridge", vs2),
]
best_head = sorted(choices, key=lambda x: x[1], reverse=True)[0][0]

print("best baseline:", best)
print(f"VS base={vs_base:.6f} | base_cal={vs_base_cal:.6f} | meta_pos={vs1:.6f} | meta_ridge={vs2:.6f}")
print("selected head:", best_head)

feat_df.to_csv(PRED_DIR / "oof_feature_table.csv", index=False)


In [ ]:
# ===== 测试推理 + 头部选择输出 =====
models=[]
for seed, plist in model_paths_by_seed.items():
    for fold, p in enumerate(plist):
        m = AttentionUNet3D(base=CFG["base_ch"]).to(DEVICE)
        m.load_state_dict(torch.load(p, map_location=DEVICE))
        m.eval(); models.append(m)

rows=[]
detail=[]
for tp in tqdm(all_test_imgs, desc="Test inference"):
    pid = parse_pid(tp)
    arr, sp = load_nii(tp)
    arr = robust_zscore(arr)
    arr = np.transpose(arr, (2,0,1)).astype(np.float32)
    x = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0).to(DEVICE)
    probs=[]
    with torch.no_grad():
        for m in models:
            p = infer_tta(m, x).cpu().numpy()[0,0] if CFG["tta"] else torch.sigmoid(m(x)).cpu().numpy()[0,0]
            probs.append(p)
    prob = np.mean(probs, axis=0).astype(np.float32)

    # base heads
    vh = volume_from_binary((prob > best["thr"]).astype(np.uint8), sp)
    vs = volume_from_prob(prob, sp)
    base_pred = best["alpha"] * vh + (1-best["alpha"]) * vs
    base_cal_pred = max(0.0, float(k * base_pred + b))

    # meta heads
    f = build_volume_features(prob, sp, thr_list)
    xv = np.array([[f[c] for c in feature_cols]], dtype=np.float64)
    p1 = max(0.0, float(meta1.predict(xv)[0]))
    p2 = max(0.0, float(meta2.predict(xv)[0]))

    if best_head == "base":
        final = base_pred
    elif best_head == "base_cal":
        final = base_cal_pred
    elif best_head == "meta_linear_pos":
        final = p1
    else:
        final = p2

    rows.append({"patient_id": pid, "volume": float(max(final, 0.0))})
    detail.append({
        "patient_id": pid,
        "base": float(base_pred),
        "base_cal": float(base_cal_pred),
        "meta_pos": float(p1),
        "meta_ridge": float(p2),
        "selected": best_head,
        "final": float(max(final, 0.0)),
    })

sub = pd.DataFrame(rows).sort_values("patient_id").reset_index(drop=True)
sub_detail = pd.DataFrame(detail).sort_values("patient_id").reset_index(drop=True)
sub.to_csv(PRED_DIR / "submission.csv", index=False)
sub_detail.to_csv(PRED_DIR / "submission_detail.csv", index=False)

save_paths=[
    Path("/root/setup/solution/working/submission.csv"),
    Path("/working/submission.csv"),
    OUTPUT_ROOT / "submission.csv",
]
saved=[]
for sp in save_paths:
    try:
        sp.parent.mkdir(parents=True, exist_ok=True)
        sub.to_csv(sp, index=False)
        saved.append(str(sp))
    except Exception as e:
        print("skip", sp, e)

meta = {
    "run_name": RUN_NAME,
    "cfg": CFG,
    "best_baseline": best,
    "best_head": best_head,
    "scores": {"vs_base": float(vs_base), "vs_base_cal": float(vs_base_cal), "vs_meta_pos": float(vs1), "vs_meta_ridge": float(vs2)},
    "saved_submission_paths": saved,
}
with open(LOG_DIR / "run_meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print("Saved submission to:")
for p in saved: print(" -", p)
print(sub.head())


In [ ]:
# 提交前自检
for p in [Path("/root/setup/solution/working/submission.csv"), Path("/working/submission.csv"), PRED_DIR / "submission.csv"]:
    print(p, "exists=", p.exists())
    if p.exists():
        d = pd.read_csv(p)
        print(" shape", d.shape, " cols", d.columns.tolist())
        print(d.head(3))


## 备注
- 这个版本更偏向“指标工程 + 稳健集成”，通常比单纯加深网络更稳定。
- 若平台时长不足，先把 `CFG["seeds"]` 改为 `[42, 2025]`。
